# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [1]:
import numpy as np
from pathlib import Path
import numpy as np
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cpu


In [2]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


In [3]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)}")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 20


In [4]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 1,#20,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 300,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    feature_cols = FEATURE_COLS,
)

model, scalers, val_loader, va_id_map = train_val(config)

Epoch 01 | Train: 0.618875 | Val: 374009.476103


## 4. Evaluation
Calculate implementation error (bps) usage `alpha=1.0` (pure model).

In [5]:
# # Combined Evaluation & R^2 Calculation (Robust Version)
# run_full_eval = True

# if run_full_eval and 'model' in globals():
#     from sklearn.metrics import r2_score
#     from tqdm import tqdm
#     import numpy as np
    
#     # 1. Setup Feature Indices (Matches DeepLOB interleaving: P, V, P, V)
#     LOB_COLS = []
#     for i in range(1, 6):
#         LOB_COLS.extend([f"ask_price_{i}", f"ask_vol_{i}", f"bid_price_{i}", f"bid_vol_{i}"])

#     model.eval()
    
#     # Storage
#     all_preds_z = []
#     all_targets_z = []
#     model_errs_bps = []
    
#     # Indices for Mid-Price calculation (Ask 1 and Bid 1)
#     # These are indices relative to the LOB_COLS list
#     a1_idx = 0  # ask_price_1
#     b1_idx = 2  # bid_price_1

#     # We use the raw DataFrames to iterate by ID
#     va_ids_array = va_id_map.cpu().numpy().flatten()
#     id_to_idx = {int(uid): i for i, uid in enumerate(va_ids_array)}
#     valid_eval_ids = [hid for hid in va_ids if int(hid) in id_to_idx]

#     print(f"Running robust combined eval on {len(valid_eval_ids)} instruments...")

#     for hid in tqdm(valid_eval_ids):
#         idx = id_to_idx[int(hid)]
        
#         # Get hour-specific data
#         xh_raw = x_va_raw[x_va_raw["anonymized_id"] == hid]
#         yh_raw = y_va_raw[y_va_raw["anonymized_id"] == hid]
        
#         if len(yh_raw) < config.target_window:
#             continue
            
#         # 2. Get Instrument-Specific Scalers
#         h_feat_means = scalers['feat_means'][0, idx, :].cpu().numpy()
#         h_feat_stds  = scalers['feat_stds'][0, idx, :].cpu().numpy()
        
#         # Determine Mid-Price scalers for this specific instrument
#         # We average the Ask1 and Bid1 stats to get a proxy for Mid-Price stats
#         h_mid_mean = (h_feat_means[a1_idx] + h_feat_means[b1_idx]) / 2.0
#         h_mid_std  = (h_feat_stds[a1_idx] + h_feat_stds[b1_idx]) / 2.0
        
#         # 3. Prepare Input
#         xh = xh_raw.tail(config.input_window)
#         x_arr = xh[LOB_COLS].astype(np.float32).to_numpy()
        
#         # Z-Score the input features
#         x_arr = (x_arr - h_feat_means) / (h_feat_stds + 1e-9)
#         xb = torch.from_numpy(x_arr).unsqueeze(0).to(device)
        
#         # 4. Predict
#         with torch.no_grad():
#             pred_z_batch = model(xb, y_teacher=None) 
#             pred_z = pred_z_batch.cpu().numpy().flatten() # [60]

#         # 5. Target Extraction (Z-Space)
#         yh_future = yh_raw.head(config.target_window)
#         target_mid_raw = (yh_future["ask_price_1"] + yh_future["bid_price_1"]) / 2.0
        
#         # Normalize target using the SAME instrument-specific stats
#         target_z = (target_mid_raw.to_numpy() - h_mid_mean) / (h_mid_std + 1e-9)

#         # Store for Stats
#         all_preds_z.append(pred_z)
#         all_targets_z.append(target_z)

#         # 6. Trading Simulation (Dollar Space)
#         # Convert prediction to dollars using this instrument's scale
#         mid_pred_dollars = (pred_z * h_mid_std) + h_mid_mean
#         positions = schedule_from_forecasts(mid_pred_dollars, volume_to_fill)
        
#         if positions.sum() > 0:
#             positions = positions * (volume_to_fill / positions.sum())
        
#         ask_p = yh_future[[f"ask_price_{l}" for l in range(1,6)]].to_numpy()
#         ask_v = yh_future[[f"ask_vol_{l}" for l in range(1,6)]].to_numpy()
#         bid_p = yh_future[[f"bid_price_{l}" for l in range(1,6)]].to_numpy()
#         bid_v = yh_future[[f"bid_vol_{l}" for l in range(1,6)]].to_numpy() 
        
#         total_vol, avg_p = simulate_walk_the_book(positions, ask_p, ask_v, bid_p, bid_v)
        
#         # Trading BPS calculation
#         bench_p = (ask_p[-1, 0] + bid_p[-1, 0]) / 2.0
#         if total_vol > 0 and not np.isnan(avg_p):
#             rel_err = abs(avg_p - bench_p) / bench_p
#             model_errs_bps.append(rel_err * 10000.0)

#     # --- FINAL REPORTING ---
#     print("\n" + "="*40)
#     print(f"RESULTS FOR {len(model_errs_bps)} EVALUATED HOURS")
#     print("-" * 40)

#     # Convert lists to matrices [N, 60]
#     preds_z_mat = np.vstack(all_preds_z)
#     targets_z_mat = np.vstack(all_targets_z)
    
#     # 1. Z-Space R2 (Directional accuracy at level)
#     f_preds = preds_z_mat.flatten()
#     f_targets = targets_z_mat.flatten()
#     mask = np.isfinite(f_preds) & np.isfinite(f_targets)
    
#     r2_z = r2_score(f_targets[mask], f_preds[mask])
#     corr_z = np.corrcoef(f_targets[mask], f_preds[mask])[0, 1]
    
#     print(f"Z-Space R^2:      {r2_z:.6f}")
#     print(f"Z-Space Corr:     {corr_z:.4f}")

#     # 2. Return-Space R2 (Capturing the move)
#     # We calculate diff on Z-scores directly to avoid dollar-scaling issues
#     # This measures: "Did the price move X standard deviations as predicted?"
#     pred_ret = np.diff(preds_z_mat, axis=1).flatten()
#     targ_ret = np.diff(targets_z_mat, axis=1).flatten()
    
#     ret_mask = np.isfinite(pred_ret) & np.isfinite(targ_ret)
#     r2_ret = r2_score(targ_ret[ret_mask], pred_ret[ret_mask])
#     corr_ret = np.corrcoef(targ_ret[ret_mask], pred_ret[ret_mask])[0, 1]
    
#     print(f"Returns R^2:      {r2_ret:.6f}")
#     print(f"Returns Corr:     {corr_ret:.4f}")
    
#     # 3. Trading Performance
#     if model_errs_bps:
#         print(f"Mean Trading BPS: {np.mean(model_errs_bps):.4f}")
#     print("="*40)